In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install timm peft scikit-learn -q

Mounted at /content/drive


In [ ]:
!pip install -q torchao==0.16.0
!pip install timm peft scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.1 MB/s eta 0:00:00


In [ ]:
import os, json, pickle
from pathlib import Path

SEED = 42

PROJECT_DIR = Path("/content/drive/MyDrive/Indian_Food_Research")
SPLITS_DIR  = PROJECT_DIR / "splits"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR, "| exists:", PROJECT_DIR.exists())


PROJECT_DIR: /content/drive/MyDrive/Indian_Food_Research | exists: True


In [ ]:
train_idx = pickle.load(open(SPLITS_DIR / "train_idx.pkl", "rb"))
val_idx   = pickle.load(open(SPLITS_DIR / "val_idx.pkl",   "rb"))
test_idx  = pickle.load(open(SPLITS_DIR / "test_idx.pkl",  "rb"))

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx)   & set(test_idx)) == 0
print("Split integrity check passed.")

Train: 3204  Val: 687  Test: 687
Split integrity check passed.


In [ ]:
import torch, numpy as np
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

torch.manual_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

DATASET_PATH = PROJECT_DIR / "datasets" / "CompressedDataset"
full_ds      = datasets.ImageFolder(DATASET_PATH)
num_classes  = len(full_ds.classes)
print(f"Classes: {num_classes}  Total images: {len(full_ds)}")

class TransformedSubset(Dataset):
    def __init__(self, base_ds, indices, transform):
        self.base_ds   = base_ds
        self.indices   = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        path, label = self.base_ds.samples[self.indices[i]]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


Classes: 100  Total images: 4578


In [ ]:
import torch.nn as nn
import torch.optim as optim
import timm
from peft import LoraConfig, get_peft_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def build_dinov2_lora(num_classes, r=8, alpha=16, target_modules=['qkv'], dropout=0.1):
    backbone = timm.create_model('vit_base_patch14_dinov2', pretrained=True,
                                  num_classes=num_classes, img_size=224)
    config = LoraConfig(r=r, lora_alpha=alpha, target_modules=target_modules,
                        lora_dropout=dropout, modules_to_save=["head"])
    model = get_peft_model(backbone, config)
    for n, p in model.named_parameters():
        if 'head' in n or 'classifier' in n:
            p.requires_grad = True
    return model

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total

def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-4):
    model = model.to(DEVICE)
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
    best_val_acc, best_state = 0.0, None

    for epoch in range(epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        val_acc = evaluate_model(model, val_loader)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'    epoch {epoch+1}/{epochs}  val_acc={val_acc:.4f}')

    if best_state:
        model.load_state_dict(best_state)
    return model, best_val_acc

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total


Device: cuda


In [ ]:
full_train_loader = DataLoader(TransformedSubset(full_ds, train_idx, train_tf),
                               batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader        = DataLoader(TransformedSubset(full_ds, val_idx, eval_tf),
                               batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader       = DataLoader(TransformedSubset(full_ds, test_idx, eval_tf),
                               batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import time
t = time.time()
imgs, labels = next(iter(full_train_loader))
print(f"One batch loaded in {time.time()-t:.1f}s  shape={imgs.shape}")

One batch loaded in 24.9s  shape=torch.Size([32, 3, 224, 224])


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   47G   66G  42% /


In [ ]:
import shutil

ORIGINAL_DS = Path("/content/drive/MyDrive/Indian_Food_Research/datasets/CompressedDataset")
LOCAL_DS = Path("/content/indian_food_cached")

if LOCAL_DS.exists():
    shutil.rmtree(str(LOCAL_DS))

print("Copying...")
shutil.copytree(str(ORIGINAL_DS), str(LOCAL_DS))
print("Done.")

full_ds = datasets.ImageFolder(str(LOCAL_DS))
num_classes = len(full_ds.classes)
print(f"Classes: {num_classes}  Total: {len(full_ds)}")

Copying...
Done.
Classes: 100  Total: 4578


In [ ]:
ablation_configs = [
    {'r': 4,  'target_modules': ['qkv']},
    {'r': 8,  'target_modules': ['qkv']},
    {'r': 16, 'target_modules': ['qkv']},
    {'r': 4,  'target_modules': ['qkv', 'proj']},
    {'r': 8,  'target_modules': ['qkv', 'proj']},
    {'r': 16, 'target_modules': ['qkv', 'proj']},
]

CKPT_PATH = str(RESULTS_DIR / 'ablation_checkpoints.json')

if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        ablation_results = json.load(f)
    done_tags = {r['config'] for r in ablation_results}
    print(f"Resuming — {len(done_tags)}/6 configs already done: {done_tags}")
else:
    ablation_results, done_tags = [], set()

for cfg in ablation_configs:
    tag = f"r{cfg['r']}_" + '+'.join(cfg['target_modules'])

    if tag in done_tags:
        print(f"SKIP (done): {tag}")
        continue

    print(f'\n=== LoRA ablation: {tag} ===')
    model = build_dinov2_lora(num_classes, r=cfg['r'], target_modules=cfg['target_modules'])
    trainable, total = count_trainable_params(model)

    model, best_val_acc = train_model(model, full_train_loader, val_loader, epochs=10, lr=1e-3)
    test_acc = evaluate_model(model, test_loader)

    print(f'  trainable: {trainable:,} ({100*trainable/total:.3f}%)  test_acc: {test_acc:.4f}')

    ablation_results.append({
        'config': tag,
        'lora_rank': cfg['r'],
        'target_modules': '+'.join(cfg['target_modules']),
        'trainable_params': trainable,
        'trainable_pct': 100 * trainable / total,
        'val_acc': float(best_val_acc),
        'test_acc': float(test_acc)
    })

    # Checkpoint after EVERY config
    with open(CKPT_PATH, 'w') as f:
        json.dump(ablation_results, f, indent=2)
    done_tags.add(tag)
    print(f'  Checkpoint saved ({len(ablation_results)}/6 done)')

    del model
    torch.cuda.empty_cache()

Resuming — 6/6 configs already done: {'r4_qkv+proj', 'r16_qkv+proj', 'r4_qkv', 'r16_qkv', 'r8_qkv+proj', 'r8_qkv'}
SKIP (done): r4_qkv
SKIP (done): r8_qkv
SKIP (done): r16_qkv
SKIP (done): r4_qkv+proj
SKIP (done): r8_qkv+proj
SKIP (done): r16_qkv+proj


In [ ]:
import pandas as pd

ablation_df = pd.DataFrame(ablation_results)

le_csv = str(RESULTS_DIR / 'label_efficiency_results.csv')
le_df  = pd.read_csv(le_csv)

ref_rows = (le_df[le_df['fraction'] == 1.0]
            [['method', 'trainable_params', 'trainable_pct', 'val_acc', 'test_acc']]
            .rename(columns={'method': 'config'}))

table6_df = pd.concat([ablation_df, ref_rows], ignore_index=True)
table6_path = str(RESULTS_DIR / 'table6_lora_ablation.csv')
table6_df.to_csv(table6_path, index=False)
print(f'Table 6 saved: {table6_path}')
table6_df

Table 6 saved: /content/drive/MyDrive/Indian_Food_Research/results/table6_lora_ablation.csv


,config,lora_rank,target_modules,trainable_params,trainable_pct,val_acc,test_acc
0,r4_qkv,4.0,qkv,224356,0.261033,0.956332,0.960699
1,r8_qkv,8.0,qkv,371812,0.431854,0.960699,0.947598
2,r16_qkv,16.0,qkv,666724,0.771746,0.962154,0.953421
3,r4_qkv+proj,4.0,qkv+proj,303508,0.352800,0.954876,0.951965
4,r8_qkv+proj,8.0,qkv+proj,530116,0.614591,0.960699,0.954876
5,r16_qkv+proj,16.0,qkv+proj,983332,1.134070,0.962154,0.949054
6,resnet50_scratch,NaN,NaN,23712932,100.000000,0.609898,0.592431
7,linear_probe,NaN,NaN,76900,0.089625,0.957787,0.944687
8,full_finetune,NaN,NaN,85801828,100.000000,0.947598,0.946143
9,lora,NaN,NaN,448712,0.520707,0.959243,0.959243


In [ ]:
print('results/:', os.listdir(str(RESULTS_DIR)))

results/: ['resolution_statistics.csv', 'brightness_statistics.csv', 'contrast_statistics.csv', 'blur_statistics.csv', 'entropy_statistics.csv', 'quality_metrics.csv', 'test_accuracy.csv', 'confusion_matrix.csv', 'per_class_accuracy.csv', 'training_history.csv', '.ipynb_checkpoints', 'table2_cultural_gap.csv', 'top_confusion_pairs.csv', 'top_confusion_pairs (2).gsheet', 'ablation_checkpoints.json', 'top_confusion_pairs (1).gsheet', 'top_confusion_pairs.gsheet', 'table6_lora_ablation.csv', 'label_efficiency_results.csv', 'label_efficiency_checkpoints.json']
